## Question 1

Which year had the highest number of additions (starting from 2020)?


Using the list of S&P 500 companies from Wikipedia's S&P 500 companies page(https://en.wikipedia.org/wiki/List_of_S%26P_500_companies), download the data including the year each company was added to the index.

Hint: you can use pandas.read_html to scrape the data into a DataFrame. You may need to add headers to make it work:

teps:

Create a DataFrame with company tickers, names, and the year they were added.
Extract the year from the addition date and calculate the number of stocks added each year.
Which (full) year had the highest number of additions, starting from 2020?

Additional: How many current S&P 500 stocks have been in the index for more than 20 years?

In [4]:
import pandas as pd
from datetime import datetime
import requests
from io import StringIO

# Scrape S&P 500 companies data from Wikipedia
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()
tables = pd.read_html(StringIO(response.text), header=0)
df = tables[0]

# Display first few rows to understand the structure
print(df.head())
print(df.columns)



  Symbol             Security             GICS Sector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry    Headquarters Location  Date added  \
0        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04   
1               Building Products     Milwaukee, Wisconsin  2017-07-26   
2           Health Care Equipment  North Chicago, Illinois  1957-03-04   
3                   Biotechnology  North Chicago, Illinois  2012-12-31   
4  IT Consulting & Other Services          Dublin, Ireland  2011-07-06   

       CIK      Founded  
0    66740         1902  
1    91142         1916  
2     1800         1888  
3  1551152  2013 (1888)  
4  1467373         1989  
Index(['Symbol', 'Security', 'GICS Sector', 'GICS Sub-In

In [5]:
# Extract year from 'Date added' column
df['Year Added'] = pd.to_datetime(df['Date added']).dt.year

# Filter for years 2020 onwards
df_2020_onwards = df[df['Year Added'] >= 2020]

# Count additions per year
additions_per_year = df_2020_onwards['Year Added'].value_counts().sort_index()
print("\nAdditions per year (2020 onwards):")
print(additions_per_year)




Additions per year (2020 onwards):
Year Added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64


In [6]:
# Find the year with highest additions
highest_year = additions_per_year.idxmax()
highest_count = additions_per_year.max()
print(f"\nYear with highest additions (2020 onwards): {highest_year} with {highest_count} additions")




Year with highest additions (2020 onwards): 2025 with 18 additions


In [7]:
# Additional: Count stocks in index for more than 20 years
current_year = datetime.now().year
df['Years in Index'] = current_year - df['Year Added']
stocks_over_20_years = (df['Years in Index'] > 20).sum()
print(f"\nS&P 500 stocks in index for more than 20 years: {stocks_over_20_years}")


S&P 500 stocks in index for more than 20 years: 218


## Question 2

How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

Using Yahoo Finance World Indices data, compare the year-to-date (YTD) performance (1 January-21 August 2026, use the closing price to calculate the growth) of major stock market indexes for the following countries:

United States - S&P 500 (^GSPC)
China - Shanghai Composite (000001.SS)
Hong Kong - HANG SENG INDEX (^HSI)
Australia - S&P/ASX 200 (^AXJO)
India - Nifty 50 (^NSEI)
Canada - S&P/TSX Composite (^GSPTSE)
Germany - DAX (^GDAXI)
United Kingdom - FTSE 100 (^FTSE)
Japan - Nikkei 225 (^N225)
Mexico - IPC Mexico (^MXX)
Brazil - Ibovespa (^BVSP)

Additional: How many of these indexes have better returns than the S&P 500 over 3, 5, and 10 year periods? Do you see the same trend? Note: For simplicity, ignore currency conversion effects.

In [9]:
import pandas as pd
import yfinance as yf

end_date = pd.Timestamp("2026-08-21")

symbols = {
    "^GSPC": "US (S&P 500)",
    "000001.SS": "China (Shanghai Composite)",
    "^HSI": "Hong Kong (Hang Seng)",
    "^AXJO": "Australia (ASX 200)",
    "^NSEI": "India (Nifty 50)",
    "^GSPTSE": "Canada (TSX Composite)",
    "^GDAXI": "Germany (DAX)",
    "^FTSE": "UK (FTSE 100)",
    "^N225": "Japan (Nikkei 225)",
    "^MXX": "Mexico (IPC)",
    "^BVSP": "Brazil (Ibovespa)",
}

def period_return(hist, years):
    target = end_date - pd.DateOffset(years=years)
    start_slice = hist[hist.index >= target]
    if start_slice.empty:
        return float("nan")
    start_price = start_slice.iloc[0]["Close"]
    end_price = hist["Close"].iloc[-1]
    return end_price / start_price - 1

results = []

for symbol, name in symbols.items():
    hist = yf.download(
        symbol,
        start="2016-01-01",
        end=end_date + pd.Timedelta(days=1),
        auto_adjust=False,
        progress=False,
        actions=False,
    )
    if hist.empty:
        continue

    hist = hist[["Close"]].dropna()
    
    # YTD return: 1 Jan 2026 through Aug 21 2026
    ytd_start = pd.Timestamp("2026-01-01")
    ytd_slice = hist[hist.index >= ytd_start]
    if ytd_slice.empty:
        ytd = float("nan")
    else:
        ytd = hist["Close"].iloc[-1] / ytd_slice.iloc[0]["Close"] - 1

    results.append({
        "Symbol": symbol,
        "Index": name,
        "YTD Return": ytd,
        "3Y Return": period_return(hist, 3),
        "5Y Return": period_return(hist, 5),
        "10Y Return": period_return(hist, 10),
    })

df = pd.DataFrame(results)

# Compare each index against the US (S&P 500) among the non-US indexes
us_ytd = df.loc[df["Symbol"] == "^GSPC", "YTD Return"].iat[0]
non_us = df[df["Symbol"] != "^GSPC"]

def as_scalar(x):
    if hasattr(x, "item"):
        try:
            return x.item()
        except Exception:
            pass
    if hasattr(x, "squeeze"):
        x = x.squeeze()
        if hasattr(x, "item"):
            try:
                return x.item()
            except Exception:
                pass
    return x

for col in ["YTD Return", "3Y Return", "5Y Return", "10Y Return"]:
    df[col] = df[col].apply(as_scalar)

us_ytd = float(df.loc[df["Symbol"] == "^GSPC", "YTD Return"].iat[0])
non_us = df[df["Symbol"] != "^GSPC"]
ytd_better = int((non_us["YTD Return"] > us_ytd).sum())
print(f"Indexes with better YTD return than the S&P 500: {ytd_better} out of {len(non_us)}")
print("\nYTD return comparison:")
print(df.sort_values("YTD Return", ascending=False).reset_index(drop=True))

for years in [3, 5, 10]:
    col = f"{years}Y Return"
    us_val = df.loc[df["Symbol"] == "^GSPC", col].iat[0]
    better = int((non_us[col] > us_val).sum())
    print(f"\nIndexes with better {years}-year return than the S&P 500: {better} out of {len(non_us)}")

Indexes with better YTD return than the S&P 500: 2 out of 10

YTD return comparison:
       Symbol                       Index  YTD Return  3Y Return  5Y Return  \
0       ^N225          Japan (Nikkei 225)    0.273641   1.091399   1.401098   
1     ^GSPTSE      Canada (TSX Composite)    0.148566   0.850917   0.788331   
2       ^GSPC                US (S&P 500)    0.118962   0.744266   0.713209   
3       ^FTSE               UK (FTSE 100)    0.086975   0.490341   0.521536   
4       ^BVSP           Brazil (Ibovespa)    0.065361   0.494656   0.455938   
5      ^GDAXI               Germany (DAX)    0.065088   0.675068   0.648704   
6       ^AXJO         Australia (ASX 200)    0.037936   0.273122   0.209482   
7        ^MXX                Mexico (IPC)    0.024755   0.237610   0.262696   
8        ^HSI       Hong Kong (Hang Seng)   -0.012492   0.475857   0.035838   
9   000001.SS  China (Shanghai Composite)   -0.029382   0.262602   0.123111   
10      ^NSEI            India (Nifty 50)   -0

## Question 3

Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

For this task, define a correction as an event when a stock index goes down by at least 5% from the most recent all-time high.

Steps:

Download S&P 500 historical data (period from 1950 to present) using yfinance (daily stats)
Identify all-time high closing price points (where price exceeds all previous days' prices)
For each pair of consecutive all-time highs, find the minimum price in between
Calculate drawdown percentages: (high - low) / high × 100
Filter for corrections with at least 5% drawdown
Calculate the duration in days for each correction period
Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns

In [10]:
import pandas as pd
import yfinance as yf

# Download S&P 500 daily history from 1950 to present
sp500_raw = yf.download("^GSPC", start="1950-01-01", progress=False, auto_adjust=False, actions=False)

# Handle possible single/multi-index output from yfinance
if isinstance(sp500_raw.columns, pd.MultiIndex):
    sp500_close = sp500_raw["Close"].iloc[:, 0].dropna()
else:
    sp500_close = sp500_raw["Close"].dropna()

# Identify new all-time highs (strictly above all previous closes)
running_max = sp500_close.cummax()
new_ath_mask = sp500_close > running_max.shift(1)
new_ath_mask.iloc[0] = True
ath = sp500_close[new_ath_mask]

# Measure each correction between consecutive all-time highs
records = []
ath_dates = ath.index.to_list()

for i in range(len(ath_dates) - 1):
    start_date = ath_dates[i]
    end_date = ath_dates[i + 1]
    start_price = sp500_close.loc[start_date]
    interval = sp500_close.loc[start_date:end_date]
    low_price = interval.min()
    low_date = interval.idxmin()
    drawdown_pct = (start_price - low_price) / start_price * 100
    duration_days = (end_date - start_date).days

    if drawdown_pct >= 5:
        records.append({
            "Start Date": start_date,
            "End Date": end_date,
            "Low Date": low_date,
            "Start Price": start_price,
            "Low Price": low_price,
            "Drawdown %": drawdown_pct,
            "Duration Days": duration_days,
        })

corrections = pd.DataFrame(records)

print(f"Significant corrections found: {len(corrections)}")
if not corrections.empty:
    print("\nMedian drawdown (%):", round(corrections["Drawdown %"].median(), 2))

    drawdown_q = corrections["Drawdown %"].quantile([0.25, 0.5, 0.75])
    duration_q = corrections["Duration Days"].quantile([0.25, 0.5, 0.75])

    print("\nDrawdown percentiles (%):")
    print(drawdown_q.round(2))

    print("\nDuration percentiles (days):")
    print(duration_q.round(0).astype(int))

    print("\nCorrection summary:")
    display(corrections.sort_values("Drawdown %", ascending=False).reset_index(drop=True))
else:
    print("No corrections meeting the 5% threshold were found.")

Significant corrections found: 74

Median drawdown (%): 7.99

Drawdown percentiles (%):
0.25     6.23
0.50     7.99
0.75    14.02
Name: Drawdown %, dtype: float64

Duration percentiles (days):
0.25     56
0.50     92
0.75    212
Name: Duration Days, dtype: int64

Correction summary:


,Start Date,End Date,Low Date,Start Price,Low Price,Drawdown %,Duration Days
0,2007-10-09,2013-03-28,2009-03-09,1565.150024,676.530029,56.775388,1997
1,2000-03-24,2007-05-30,2002-10-09,1527.459961,776.760010,49.146948,2623
2,1973-01-11,1980-07-17,1974-10-03,120.239998,62.279999,48.203593,2744
3,1968-11-29,1972-03-06,1970-05-26,108.370003,69.290001,36.061641,1193
4,2020-02-19,2020-08-18,2020-03-23,3386.149902,2237.399902,33.924960,181
...,...,...,...,...,...,...,...
69,2021-09-02,2021-10-21,2021-10-04,4536.950195,4300.459961,5.212538,49
70,1972-08-14,1972-11-01,1972-10-16,112.550003,106.769997,5.135501,79
71,2025-10-28,2025-12-11,2025-11-20,6890.890137,6538.759766,5.110085,44
72,1999-04-12,1999-04-22,1999-04-19,1358.630005,1289.479980,5.089688,10


In [11]:
import pandas as pd

reference = pd.DataFrame(
    [
        ("2007-10-09", "2009-03-09", 56.8, 517),
        ("2000-03-24", "2002-10-09", 49.1, 929),
        ("1973-01-11", "1974-10-03", 48.2, 630),
        ("1968-11-29", "1970-05-26", 36.1, 543),
        ("2020-02-19", "2020-03-23", 33.9, 33),
        ("1987-08-25", "1987-12-04", 33.5, 101),
        ("1961-12-12", "1962-06-26", 28.0, 196),
        ("1980-11-28", "1982-08-12", 27.1, 622),
        ("2022-01-03", "2022-10-12", 25.4, 282),
        ("1966-02-09", "1966-10-07", 22.2, 240),
    ],
    columns=["Start Date", "End Date", "Ref Drawdown %", "Ref Duration Days"],
)

reference["Start Date"] = pd.to_datetime(reference["Start Date"])
reference["End Date"] = pd.to_datetime(reference["End Date"])

top10 = corrections.sort_values("Drawdown %", ascending=False).head(10).copy()

comparison = reference.merge(
    top10,
    on=["Start Date", "End Date"],
    how="left",
    suffixes=("_ref", "_calc"),
)

comparison["Drawdown Diff"] = comparison["Drawdown %"] - comparison["Ref Drawdown %"]
comparison["Duration Diff"] = comparison["Duration Days"] - comparison["Ref Duration Days"]

print("Comparison against the provided top 10 corrections:")
print(
    comparison[
        [
            "Start Date",
            "End Date",
            "Ref Drawdown %",
            "Drawdown %",
            "Drawdown Diff",
            "Ref Duration Days",
            "Duration Days",
            "Duration Diff",
        ]
    ].to_string(index=False)
)

missing = comparison["Drawdown %"].isna().sum()
print(f"\nMatched {len(comparison) - missing}/10 listed corrections in the computed results.")

print("\nComputed top 10 corrections by drawdown:")
display(
    top10[
        [
            "Start Date",
            "End Date",
            "Drawdown %",
            "Duration Days",
        ]
    ].reset_index(drop=True)
)

Comparison against the provided top 10 corrections:
Start Date   End Date  Ref Drawdown %  Drawdown %  Drawdown Diff  Ref Duration Days  Duration Days  Duration Diff
2007-10-09 2009-03-09            56.8         NaN            NaN                517            NaN            NaN
2000-03-24 2002-10-09            49.1         NaN            NaN                929            NaN            NaN
1973-01-11 1974-10-03            48.2         NaN            NaN                630            NaN            NaN
1968-11-29 1970-05-26            36.1         NaN            NaN                543            NaN            NaN
2020-02-19 2020-03-23            33.9         NaN            NaN                 33            NaN            NaN
1987-08-25 1987-12-04            33.5         NaN            NaN                101            NaN            NaN
1961-12-12 1962-06-26            28.0         NaN            NaN                196            NaN            NaN
1980-11-28 1982-08-12            27.

,Start Date,End Date,Drawdown %,Duration Days
0,2007-10-09,2013-03-28,56.775388,1997
1,2000-03-24,2007-05-30,49.146948,2623
2,1973-01-11,1980-07-17,48.203593,2744
3,1968-11-29,1972-03-06,36.061641,1193
4,2020-02-19,2020-08-18,33.924960,181
5,1987-08-25,1989-07-26,33.509515,701
6,1961-12-12,1963-09-03,27.973568,630
7,1980-11-28,1982-11-03,27.113582,705
8,2022-01-03,2024-01-19,25.425097,746
9,1966-02-09,1967-05-04,22.177335,449


## Question 4

Earnings Surprise Analysis for Amazon (AMZN)
Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

You should have 25 entries starting from 2020-10-29. Note: the 1 future earnings date entry will not have Reported EPS
and Surprise % filled in.

Download complete historical price data using yfinance
Calculate 2-day percentage changes for all historical dates: for each sequence of 3 consecutive trading days (Day 1, Day 2, Day 3), compute the return as Close_Day3 / Close_Day1 - 1. (Assume Day 2 may correspond to the earnings announcement.)
Filter for positive earnings surprises and calculate the median 2-day return. Then calculate the correlation between the 2-day stock return and the earnings surprise magnitude.
▎ Hint: you can generate all correlations at once using pd.corr().

Additional: Is there a correlation between the magnitude of the earnings surprise and the stock price reaction? Does the market react differently to earnings surprises during bull vs. bear markets?

In [12]:
amzn = yf.Ticker("AMZN")

# Earnings dates: keep the 25 entries starting from 2020-10-29
amzn_earnings = amzn.get_earnings_dates(limit=40).rename_axis("Earnings Date").reset_index()
amzn_earnings["Earnings Date"] = (
    pd.to_datetime(amzn_earnings["Earnings Date"], utc=True)
    .dt.tz_convert(None)
    .dt.normalize()
)
amzn_earnings = (
    amzn_earnings[amzn_earnings["Earnings Date"] >= pd.Timestamp("2020-10-29")]
    .sort_values("Earnings Date")
    .head(25)
    .reset_index(drop=True)
)

# Full AMZN price history
amzn_raw = yf.download("AMZN", period="max", progress=False, auto_adjust=False, actions=False)

if isinstance(amzn_raw.columns, pd.MultiIndex):
    amzn_close = amzn_raw["Close"].iloc[:, 0].dropna()
else:
    amzn_close = amzn_raw["Close"].dropna()

# 2-day return centered on each trading day:
# return for Day1 -> Day3, with Day2 as the event date
amzn_2d_return = (amzn_close.shift(-1) / amzn_close.shift(1) - 1).rename("2D Return")
amzn_2d_return.index = pd.to_datetime(amzn_2d_return.index).normalize()

# Merge earnings with returns
amzn_analysis = amzn_earnings.merge(
    amzn_2d_return.to_frame(),
    left_on="Earnings Date",
    right_index=True,
    how="left",
)

surprise_col = next(c for c in amzn_analysis.columns if "Surprise" in str(c))
reported_col = next(c for c in amzn_analysis.columns if "Reported EPS" in str(c))

amzn_analysis[surprise_col] = pd.to_numeric(amzn_analysis[surprise_col], errors="coerce")
amzn_analysis[reported_col] = pd.to_numeric(amzn_analysis[reported_col], errors="coerce")
amzn_analysis["2D Return %"] = amzn_analysis["2D Return"] * 100

# Market regime from S&P 500: Bull if above 200-day MA, else Bear
market_regime = sp500_close.rename("SP500").to_frame()
market_regime.index = pd.to_datetime(market_regime.index).normalize()
market_regime["MA200"] = market_regime["SP500"].rolling(200).mean()
market_regime["Market Regime"] = market_regime["SP500"].ge(market_regime["MA200"]).map({True: "Bull", False: "Bear"})

amzn_analysis = amzn_analysis.merge(
    market_regime[["Market Regime"]],
    left_on="Earnings Date",
    right_index=True,
    how="left",
)

# Positive surprise analysis
positive_surprises = amzn_analysis[amzn_analysis[surprise_col] > 0].copy()
median_2d_return_pct = positive_surprises["2D Return %"].median()

# Correlation between surprise magnitude and 2-day move
corr_matrix = amzn_analysis[[surprise_col, "2D Return %"]].dropna().corr()
surprise_return_corr = corr_matrix.loc[surprise_col, "2D Return %"]

# Bull vs bear comparison
regime_summary = (
    positive_surprises.groupby("Market Regime")["2D Return %"]
    .agg(["count", "median", "mean"])
    .sort_index()
)

print(f"Entries retrieved from 2020-10-29 onward: {len(amzn_analysis)}")
print(f"Median 2-day return after positive earnings surprises: {median_2d_return_pct:.2f}%")
print(f"Correlation between surprise magnitude and 2-day return: {surprise_return_corr:.3f}")

print("\nAMZN earnings surprise analysis:")
display(
    amzn_analysis[
        ["Earnings Date", "EPS Estimate", reported_col, surprise_col, "2D Return %", "Market Regime"]
    ]
)

print("\nCorrelation matrix:")
display(corr_matrix)

print("\nPositive-surprise reaction by market regime:")
display(regime_summary)

if pd.notna(surprise_return_corr):
    if abs(surprise_return_corr) < 0.2:
        print("\nInterpretation: little to no linear relationship.")
    elif surprise_return_corr > 0:
        print("\nInterpretation: larger positive surprises tend to be followed by stronger stock reactions.")
    else:
        print("\nInterpretation: larger positive surprises tend to be followed by weaker stock reactions.")

Entries retrieved from 2020-10-29 onward: 25
Median 2-day return after positive earnings surprises: 0.35%
Correlation between surprise magnitude and 2-day return: 0.219

AMZN earnings surprise analysis:


,Earnings Date,EPS Estimate,Reported EPS,Surprise(%),2D Return %,Market Regime
0,2020-10-29,0.38,0.62,64.25,-4.003764,Bull
1,2021-02-02,0.35,0.70,100.10,-0.907901,Bull
2,2021-04-29,0.47,0.79,67.30,0.257915,Bull
3,2021-07-29,0.61,0.76,23.06,-8.338937,Bull
4,2021-10-28,0.44,0.31,-31.21,-0.591301,Bull
5,2022-02-03,0.18,1.39,657.12,4.665611,Bull
6,2022-04-28,0.42,-0.38,-190.58,-10.049798,Bear
7,2022-07-28,0.12,-0.20,-273.30,11.556581,Bear
8,2022-10-27,0.21,0.28,35.53,-10.591388,Bear
9,2023-02-02,0.17,0.03,-82.20,-1.673801,Bull



Correlation matrix:


,Surprise(%),2D Return %
Surprise(%),1.000000,0.219063
2D Return %,0.219063,1.000000



Positive-surprise reaction by market regime:


,count,median,mean
Market Regime,,,
Bear,3,3.014856,-0.781820
Bull,17,0.257915,0.683283



Interpretation: larger positive surprises tend to be followed by stronger stock reactions.


## Question 5

I want to develop a trading strategy benchmarking platform that backtests and compares strategies such as Buy & Hold, Moving Average Crossover, RSI, MACD, and Momentum using historical market data. The strategies are evaluated using performance metrics like returns, Sharpe ratio, volatility, and maximum drawdown, while also analyzing their behavior in bull, bear, and sideways markets. The goal is to identify the strengths, weaknesses, and best use cases of each strategy and create a framework for ranking them based on risk-adjusted performance.

## Question 6

VIX Index (^VIX): Also known as the 'fear index,' it measures the market's expectation of future stock market volatility over the next 30 days, serving as a forward-looking sentiment indicator.

Producer Price Index for All Commodities (PPIACO): This index measures the average change in selling prices received by domestic producers for their output, acting as a crucial indicator of inflationary pressures at the producer level.

Crude Oil Prices (WTI - DCOILWTICO, Brent - DCOILBRENTEU): These prices reflect global supply and demand for oil, impacting energy costs and inflation worldwide.

Corporate Financials (NVDA financials, balance sheet): These are detailed financial statements (income statements, balance sheets) that provide deep insights into a company's historical performance, financial health, and profitability.